# 🌌 Module 6: Crossing the Spectrum - Multimodal Astronomy

Professional astronomers rarely look at an object in just one wavelength. A galaxy looks completely different in **X-Ray** (hot gas), **Optical** (stars), and **Infrared** (dust).

In this notebook, we will:
1. Download an **Optical** image (from the DSS survey).
2. Download an **Infrared** image (from the WISE mission).
3. Use **Reprojection** to align the WISE pixels to the DSS grid.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astroquery.skyview import SkyView
from reproject import reproject_interp
from astropy.visualization import make_lupton_rgb

print("🔭 Multimodal environment ready.")

## 1. Fetching Multi-Wavelength Data
We will target the **M101 (Pinwheel Galaxy)**. We'll pull a 'blue' optical frame and a 'mid-infrared' frame.

In [ ]:
target = "M101"
print(f"🛰️ Requesting multi-wavelength data for {target}...")

# Request Optical (DSS) and Infrared (WISE 12 micron)
images = SkyView.get_images(position=target, 
                            survey=['DSS2 Blue', 'WISE 12'], 
                            radius=0.2 * u.deg)

hdu_opt = images[0][0]
hdu_ir = images[1][0]

print("✅ Data received!")
print(f"📏 Optical Resolution: {hdu_opt.data.shape}")
print(f"📏 Infrared Resolution: {hdu_ir.data.shape}")

## 2. The Problem: Misalignment
Even though they are the same object, the pixels don't line up because the cameras have different orientations and resolutions.

In [ ]:
fig = plt.figure(figsize=(12, 5))

ax1 = fig.add_subplot(1, 2, 1, projection=WCS(hdu_opt.header))
ax1.imshow(hdu_opt.data, origin='lower', cmap='Blues_r')
ax1.set_title("Optical (DSS)")

ax2 = fig.add_subplot(1, 2, 2, projection=WCS(hdu_ir.header))
ax2.imshow(hdu_ir.data, origin='lower', cmap='YlOrRd')
ax2.set_title("Infrared (WISE)")

plt.show()

## 3. The Solution: WCS Reprojection
We will re-map the WISE Infrared image so it matches the Optical image's grid *perfectly* pixel-for-pixel.

In [ ]:
print("🚀 Reprojecting Infrared data to Optical grid...")
print("⏱️ This is where your CPU/GPU compute power is used!")

# Reproject IR image to Optical WCS
ir_aligned, footprint = reproject_interp(hdu_ir, hdu_opt.header)

print("✅ Reprojection complete.")

## 4. Overlaying the Spectrum
Now that they are aligned, we can overlay them to see where the dust (IR) is compared to the stars (Optical).

In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(hdu_opt.data, origin='lower', cmap='Blues_r', alpha=0.8)
plt.imshow(ir_aligned, origin='lower', cmap='YlOrRd', alpha=0.4)
plt.title(f"Multimodal View of {target}: Stars (Blue) + Dust (Orange)")
plt.show()

## 5. Scaling this to the Fleet
When processing thousands of these 'Align and Stack' jobs, we use **Dask**. 

Each worker (NUC, Pi, GPU) is given a different wavelength or a different sky tile to reproject. This is how modern surveys like the **James Webb Space Telescope** create those massive composite images!

---